# Dual-Phase Microstructure Generator — Fractal + Fuzzy Logic Edition

Procedurally generates intricate SEM BSE-like images of dual-phase steel with a
**target martensite/ferrite phase fraction**.

## Physics basis
- **Ferrite** (BCC, light phase in BSE): equiaxed polygonal Voronoi grains with
  multi-scale fractal roughness (diamond-square) and fuzzy Gaussian boundary fading
- **Martensite** (BCT, dark phase in BSE): lath sub-structure with
  fuzzy crystallographic misorientation contrast (Kurdjumov-Sachs variants),
  fractal lath taper profiles, and diamond-square texture overlay
- **Phase boundaries**: irregular fractal edges via signed-distance + DS noise
  perturbation; soft halo transitions via Gaussian membership blending

## Algorithms
| Component | Algorithm |
|---|---|
| Grain tessellation | Voronoi nearest-seed |
| Grain texture (coarse) | Orientation-gradient (channelling contrast) |
| Grain texture (fine) | Diamond-square midpoint displacement (Hurst H=0.8) |
| Phase boundary shape | SDF + diamond-square perturbation (H=0.65, smoothed) |
| Lath BSE contrast | Fuzzy 3-zone misorientation model + centroid defuzzification |
| Lath tip morphology | 1D midpoint-displacement taper (H=0.4) |
| Boundary halos | Gaussian EDT membership blending |
| Final noise | Gaussian blur + additive white noise |

## Controls
- `martensite_fraction` — target volume fraction of martensite (0.0–1.0)
- `fractal_roughness_ferrite / _martensite / _boundary` — Hurst exponents (0=rough, 1=smooth)
- `boundary_perturbation_amplitude` — max phase-boundary displacement in pixels
- `fuzzy_sigma_gb / _phase` — soft boundary transition widths in pixels

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.spatial import Voronoi
from scipy.ndimage import gaussian_filter, label as nd_label
from PIL import Image, ImageDraw, ImageFont
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

## Core generation utilities

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Utility: rasterise a Voronoi diagram onto a pixel canvas
# ──────────────────────────────────────────────────────────────────────────────

def voronoi_region_map(seeds: np.ndarray, size: int) -> np.ndarray:
    """Return an (size x size) integer array where each pixel holds
    the index of the nearest seed (i.e., Voronoi cell id)."""
    xs, ys = np.meshgrid(np.arange(size), np.arange(size), indexing='xy')
    # (size*size, 2)
    pixels = np.column_stack([xs.ravel(), ys.ravel()]).astype(np.float32)
    # Batched nearest-seed via broadcast
    diff = pixels[:, None, :] - seeds[None, :, :]   # (N_pix, N_seeds, 2)
    dist2 = (diff ** 2).sum(axis=2)                  # (N_pix, N_seeds)
    region_ids = dist2.argmin(axis=1).reshape(size, size)
    return region_ids


def grain_boundary_mask(region_map: np.ndarray, thickness: int = 1) -> np.ndarray:
    """Boolean mask of pixels that sit on a grain boundary (neighbour has
    different region id)."""
    shifted_x = np.roll(region_map, 1, axis=1)
    shifted_y = np.roll(region_map, 1, axis=0)
    boundary = (region_map != shifted_x) | (region_map != shifted_y)
    if thickness > 1:
        from scipy.ndimage import binary_dilation
        boundary = binary_dilation(boundary, iterations=thickness - 1)
    return boundary


print('Voronoi utilities defined.')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Utility: martensite lath sub-structure
# Upgraded to use:
#   • Fuzzy misorientation contrast per lath (replaces random uniform brightness)
#   • Fractal taper profile along each lath's long axis
#   • Diamond-square texture overlay inside each lath
# ──────────────────────────────────────────────────────────────────────────────

def draw_martensite_laths(
    canvas: np.ndarray,
    mask: np.ndarray,
    rng: np.random.Generator,
    n_packets: int = 4,
    laths_per_packet: int = 8,
    lath_width: float = 1.5,
    boundary_value: float = 0.05,
    ds_map: np.ndarray = None,
    ds_amplitude: float = 0.018,
    tip_roughness: float = 0.40,
) -> np.ndarray:
    """Render lath martensite sub-structure inside `mask`.

    Parameters
    ----------
    canvas          : float32 (H, W)
    mask            : bool (H, W) — True = this martensite island
    rng             : np.random.Generator
    n_packets       : int   — number of habit-plane packets per island
    laths_per_packet: int   — laths per packet
    lath_width      : float — half-width of each lath in pixels
    boundary_value  : float — sub-boundary dark value
    ds_map          : float32 (H, W) or None — fractal texture map
    ds_amplitude    : float — amplitude of DS overlay on lath interior
    tip_roughness   : float — Hurst exponent for fractal taper (0=noisy, 1=smooth)
    """
    ys, xs = np.where(mask)
    if len(ys) == 0:
        return canvas

    cx, cy = xs.mean(), ys.mean()
    grain_extent = max(int(xs.max() - xs.min()), int(ys.max() - ys.min()), 2)

    # Each packet gets a reference misorientation angle relative to the
    # parent austenite (3 Bain-group variant clusters)
    packet_misorientations = rng.uniform(0, np.pi / 3, n_packets)
    packet_angles          = rng.uniform(0, np.pi, n_packets)

    for p_theta, p_angle in zip(packet_misorientations, packet_angles):
        # Per-lath misorientations (intra-packet spread ≈ ±5°)
        lath_theta  = np.clip(
            p_theta + rng.normal(0, np.deg2rad(5.0), laths_per_packet),
            0.0, np.pi / 3
        )
        # Fuzzy defuzzification → per-lath BSE grey value
        lath_vals   = misorientation_contrast(lath_theta)

        # Lath directions within this packet (small perturbation around p_angle)
        lath_angles = p_angle + rng.normal(0, 0.05, laths_per_packet)

        for angle, val in zip(lath_angles, lath_vals):
            dx, dy = np.cos(angle), np.sin(angle)
            nx, ny = -dy, dx   # normal to lath plane

            # Random offset from grain centroid
            offset = rng.uniform(-grain_extent * 0.5, grain_extent * 0.5)

            # Signed distance of each grain pixel to this lath plane
            dist     = (xs - cx) * nx + (ys - cy) * ny - offset
            # Position along lath axis (for taper)
            pos_axis = (xs - cx) * dx + (ys - cy) * dy

            # ── Fractal taper: width varies along lath axis ───────────────
            taper = fractal_lath_taper(grain_extent, tip_roughness, rng)
            # Map axis position to taper index
            pos_norm = (pos_axis - pos_axis.min()) / (np.ptp(pos_axis) + 1e-8)
            taper_idx = np.clip(
                (pos_norm * (len(taper) - 1)).astype(int), 0, len(taper) - 1
            )
            effective_half_width = lath_width * taper[taper_idx]

            in_lath  = np.abs(dist) < effective_half_width
            in_core  = np.abs(dist) < (effective_half_width * 0.5)

            if not in_lath.any():
                continue

            lath_mask = np.zeros(mask.shape, dtype=bool)
            lath_mask[ys[in_lath], xs[in_lath]] = True
            lath_mask &= mask

            core_mask = np.zeros(mask.shape, dtype=bool)
            core_mask[ys[in_core], xs[in_core]] = True
            core_mask &= mask

            # Paint lath interior
            ly, lx = np.where(lath_mask)
            canvas[ly, lx] = val

            # ── DS fractal texture overlay ────────────────────────────────
            if ds_map is not None and len(ly) > 0:
                canvas[ly, lx] = np.clip(
                    canvas[ly, lx] + (ds_map[ly, lx] - 0.5) * ds_amplitude,
                    0.0, 1.0
                )

            # Sub-boundary (edge of lath, not in core)
            sub_boundary = lath_mask & ~core_mask
            canvas[sub_boundary] = boundary_value

    return canvas


print('draw_martensite_laths (misorientation fuzzy + fractal taper + DS) defined.')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Utility: ferrite grain interior texture
# Now uses:
#   • Diamond-square fractal map for multi-scale roughness
#   • Fuzzy Gaussian membership fade toward grain boundaries
#   • Orientation-gradient (channelling contrast) as before
# ──────────────────────────────────────────────────────────────────────────────

def ferrite_grain_texture(
    canvas: np.ndarray,
    mask: np.ndarray,
    base_brightness: float,
    rng: np.random.Generator,
    texture_amplitude: float = 0.04,
    scratch_prob: float = 0.3,
    ds_map: np.ndarray = None,
    ds_amplitude: float = 0.025,
    edt_global: np.ndarray = None,
    sigma_inner: float = 4.0,
    gb_dark: float = 0.05,
) -> np.ndarray:
    """Fill a ferrite grain with physically-motivated BSE texture.

    Parameters
    ----------
    canvas           : float32 (H, W) — image canvas (modified in-place)
    mask             : bool (H, W)    — True = this grain's pixels
    base_brightness  : float          — mean BSE grey for this grain
    rng              : np.random.Generator
    texture_amplitude: float          — coarse orientation-gradient amplitude
    scratch_prob     : float          — probability of a slip/scratch line
    ds_map           : float32 (H, W) or None — pre-generated DS fractal map
    ds_amplitude     : float          — fine fractal roughness amplitude
    edt_global       : float32 (H, W) or None — EDT distance from all boundaries
    sigma_inner      : float          — fuzzy fade half-width at grain boundary (px)
    gb_dark          : float          — boundary dark value for fuzzy blend
    """
    ys, xs = np.where(mask)
    if len(ys) == 0:
        return canvas

    # ── Orientation-gradient (coarse channelling contrast) ────────────────────
    angle    = rng.uniform(0, np.pi)
    gradient = np.cos(angle) * xs + np.sin(angle) * ys
    g_norm   = (gradient - gradient.min()) / (np.ptp(gradient) + 1e-8)
    brightness = base_brightness + (g_norm - 0.5) * texture_amplitude

    # ── Fractal roughness overlay (fine scale) ────────────────────────────────
    if ds_map is not None:
        brightness = brightness + (ds_map[ys, xs] - 0.5) * ds_amplitude

    # ── Fuzzy boundary fade (Gaussian membership) ────────────────────────────
    if edt_global is not None:
        d = edt_global[ys, xs]
        mu = 1.0 - np.exp(-(d ** 2) / (2.0 * sigma_inner ** 2 + 1e-8))
        brightness = mu * brightness + (1.0 - mu) * gb_dark

    canvas[ys, xs] = np.clip(brightness, 0.0, 1.0)

    # ── Occasional slip / scratch lines ──────────────────────────────────────
    if rng.random() < scratch_prob:
        scratch_angle = angle + rng.normal(0, 0.1)
        snx = -np.sin(scratch_angle)
        sny =  np.cos(scratch_angle)
        cx, cy = xs.mean(), ys.mean()
        offset = rng.uniform(-10, 10)
        dist   = np.abs((xs - cx) * snx + (ys - cy) * sny - offset)
        in_scratch = dist < 0.8
        # Only scratch where fuzzy membership is high (away from boundary)
        if edt_global is not None:
            in_scratch &= (edt_global[ys, xs] > sigma_inner * 0.3)
        canvas[ys[in_scratch], xs[in_scratch]] = np.clip(
            canvas[ys[in_scratch], xs[in_scratch]] - 0.06, 0.0, 1.0
        )

    return canvas


print('ferrite_grain_texture (fractal + fuzzy) defined.')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Fuzzy Helper 1: Misorientation-driven lath contrast (fuzzy defuzzification)
#
# Maps crystallographic misorientation angle(s) to BSE grey values using
# three overlapping fuzzy membership zones (low / medium / high misorientation)
# and centroid defuzzification.
#
# Zones (simplified from 24 Kurdjumov-Sachs variants):
#   Low  (0–15°)  → bright lath    ≈ 0.22
#   Med  (5–45°)  → mid contrast   ≈ 0.15
#   High (35–60°) → dark lath      ≈ 0.08
# ──────────────────────────────────────────────────────────────────────────────

_LOW_CUT   = np.deg2rad(15.0)
_MED_A     = np.deg2rad(5.0)
_MED_B     = np.deg2rad(22.0)
_MED_C     = np.deg2rad(45.0)
_HIGH_START= np.deg2rad(35.0)
_HIGH_MAX  = np.deg2rad(60.0)

_BRIGHT_LOW  = 0.22
_BRIGHT_MED  = 0.15
_BRIGHT_HIGH = 0.08


def _triangular_mf(x, a, b, c):
    """Triangular membership: rises linearly a→b, falls linearly b→c."""
    left  = np.where(x <= a, 0.0, (x - a) / (b - a + 1e-9))
    right = np.where(x >= c, 0.0, (c - x) / (c - b + 1e-9))
    return np.clip(np.minimum(left, right), 0.0, 1.0)


def misorientation_contrast(theta: np.ndarray) -> np.ndarray:
    """Convert misorientation angle(s) [radians] to BSE grey value(s).

    Parameters
    ----------
    theta : scalar or array — misorientation in radians, range [0, π/3]

    Returns
    -------
    contrast : same shape as theta, float32 in [0.08, 0.22]
    """
    theta = np.asarray(theta, dtype=np.float64)

    mu_low  = np.clip(1.0 - theta / (_LOW_CUT + 1e-9), 0.0, 1.0)
    mu_med  = _triangular_mf(theta, _MED_A, _MED_B, _MED_C)
    mu_high = np.clip(
        (theta - _HIGH_START) / (_HIGH_MAX - _HIGH_START + 1e-9), 0.0, 1.0
    )

    total = mu_low + mu_med + mu_high + 1e-7   # guard
    contrast = (
        (mu_low  / total) * _BRIGHT_LOW  +
        (mu_med  / total) * _BRIGHT_MED  +
        (mu_high / total) * _BRIGHT_HIGH
    )
    return contrast.astype(np.float32)


print('misorientation_contrast defined.')

In [ ]:
def generate_microstructure(
    martensite_fraction: float = 0.30,
    n_ferrite_grains: int = 40,
    n_austenite_grains: int = 20,
    image_size: int = 512,
    seed: int = 42,
    noise_sigma: float = 3.0,
    detector_noise: float = 0.015,
    boundary_thickness: int = 2,
    magnification: str = '5000x',
    show_scalebar: bool = True,
    return_phase_map: bool = False,
    # ── Fractal parameters ─────────────────────────────────────────────────
    fractal_roughness_ferrite: float = 0.80,
    fractal_roughness_martensite: float = 0.55,
    fractal_roughness_boundary: float = 0.65,
    boundary_perturbation_amplitude: float = 5.0,
    fractal_tip_roughness: float = 0.40,
    # ── Fuzzy parameters ───────────────────────────────────────────────────
    fuzzy_sigma_gb: float = 3.5,
    fuzzy_sigma_phase: float = 4.0,
):
    """
    Generate a synthetic SEM BSE-like image of dual-phase martensite/ferrite steel.

    Uses fractal diamond-square heightmaps for multi-scale grain texture and
    boundary perturbation, fuzzy misorientation-contrast for lath rendering,
    fractal lath taper profiles, and Gaussian-membership boundary blending.

    Parameters
    ----------
    martensite_fraction               : float — target M volume fraction (0–1)
    n_ferrite_grains                  : int
    n_austenite_grains                : int
    image_size                        : int   — output pixels (square)
    seed                              : int
    noise_sigma                       : float — Gaussian blur sigma (SEM DoF)
    detector_noise                    : float — additive white noise amplitude
    boundary_thickness                : int   — (unused; fuzzy blend replaces hard lines)
    magnification                     : str   — scale bar label
    show_scalebar                     : bool
    return_phase_map                  : bool
    fractal_roughness_ferrite         : float — DS Hurst exponent for ferrite texture
    fractal_roughness_martensite      : float — DS Hurst exponent for martensite texture
    fractal_roughness_boundary        : float — DS Hurst exponent for boundary noise
    boundary_perturbation_amplitude   : float — max phase-boundary displacement (px)
    fractal_tip_roughness             : float — Hurst exponent for lath taper
    fuzzy_sigma_gb                    : float — ferrite GB transition width (px)
    fuzzy_sigma_phase                 : float — martensite edge transition width (px)

    Returns
    -------
    img_array      : np.ndarray (H, W) float32, [0, 1]
    phase_map      : np.ndarray (H, W) bool   [only if return_phase_map=True]
    actual_fraction: float
    """
    assert 0.0 <= martensite_fraction <= 1.0
    rng = np.random.default_rng(seed)
    S   = image_size

    # ── Step 0.5: Generate global fractal heightmaps ──────────────────────────
    ds_ferrite    = diamond_square(S, fractal_roughness_ferrite,    rng)
    ds_martensite = diamond_square(S, fractal_roughness_martensite, rng)
    ds_boundary   = gaussian_filter(
        diamond_square(S, fractal_roughness_boundary, rng), sigma=8.0
    )

    # ── Step 1: Ferrite grain tessellation ────────────────────────────────────
    ferrite_seeds  = rng.uniform(0, S, (n_ferrite_grains, 2))
    ferrite_map    = voronoi_region_map(ferrite_seeds, S)
    ferrite_brightness = rng.uniform(0.55, 0.75, n_ferrite_grains)

    canvas = np.zeros((S, S), dtype=np.float32)

    # Global EDT from ALL ferrite grain boundaries (passed into each grain)
    gb_all    = grain_boundary_mask(ferrite_map, thickness=1)
    edt_global = distance_transform_edt(~gb_all).astype(np.float32)

    for gid in range(n_ferrite_grains):
        grain_mask = ferrite_map == gid
        canvas = ferrite_grain_texture(
            canvas, grain_mask, ferrite_brightness[gid], rng,
            ds_map=ds_ferrite,
            edt_global=edt_global,
            sigma_inner=fuzzy_sigma_gb,
        )

    # ── Step 2: Martensite island seed placement ──────────────────────────────
    gb_mask  = grain_boundary_mask(ferrite_map, thickness=4)
    gb_ys, gb_xs = np.where(gb_mask)

    if len(gb_ys) < n_austenite_grains:
        aus_seeds = rng.uniform(0, S, (n_austenite_grains, 2))
    else:
        idx    = rng.choice(len(gb_ys), n_austenite_grains, replace=False)
        jitter = rng.normal(0, S * 0.04, (n_austenite_grains, 2))
        aus_seeds = np.column_stack([gb_xs[idx], gb_ys[idx]]).astype(float) + jitter
        aus_seeds = np.clip(aus_seeds, 0, S - 1)

    aus_map = voronoi_region_map(aus_seeds, S)

    # ── Step 3: Phase assignment (greedy by grain size) ───────────────────────
    aus_grain_sizes = np.array([(aus_map == i).sum() for i in range(n_austenite_grains)])
    sorted_ids  = np.argsort(-aus_grain_sizes)
    total_pixels  = S * S
    target_pixels = int(martensite_fraction * total_pixels)

    martensite_phase = np.zeros((S, S), dtype=bool)
    accumulated = 0

    for gid in sorted_ids:
        if accumulated >= target_pixels:
            break
        grain_mask = aus_map == gid
        pixel_ys, pixel_xs = np.where(grain_mask)
        remaining = target_pixels - accumulated
        if len(pixel_ys) <= remaining:
            martensite_phase[grain_mask] = True
            accumulated += len(pixel_ys)
        else:
            cx2, cy2   = pixel_xs.mean(), pixel_ys.mean()
            cut_angle  = rng.uniform(0, np.pi)
            dx2, dy2   = np.cos(cut_angle), np.sin(cut_angle)
            proj       = (pixel_xs - cx2) * dx2 + (pixel_ys - cy2) * dy2
            order      = np.argsort(proj)
            chosen     = order[:remaining]
            martensite_phase[pixel_ys[chosen], pixel_xs[chosen]] = True
            accumulated += remaining
            break

    # ── Step 3.5: Fractal boundary perturbation ───────────────────────────────
    martensite_phase = fractal_boundary_perturbation(
        martensite_phase, ds_boundary,
        amplitude=boundary_perturbation_amplitude,
    )
    actual_fraction = martensite_phase.sum() / total_pixels

    # ── Step 4: Martensite lath sub-structure ─────────────────────────────────
    labeled_mart, n_islands = nd_label(martensite_phase)

    for island_id in range(1, n_islands + 1):
        island_mask = labeled_mart == island_id
        if island_mask.sum() < 20:
            canvas[island_mask] = rng.uniform(0.12, 0.22)
            continue
        base_val = rng.uniform(0.12, 0.28)
        canvas[island_mask] = base_val
        canvas = draw_martensite_laths(
            canvas, island_mask, rng,
            ds_map=ds_martensite,
            tip_roughness=fractal_tip_roughness,
        )

    # ── Step 5: Fuzzy boundary blend (replaces hard GB lines) ────────────────
    canvas = fuzzy_boundary_blend(
        canvas, ferrite_map, martensite_phase,
        sigma_gb=fuzzy_sigma_gb,
        sigma_phase=fuzzy_sigma_phase,
    )

    # ── Step 6: Smooth + detector noise ──────────────────────────────────────
    canvas = gaussian_filter(canvas, sigma=noise_sigma * 0.15)
    canvas += rng.normal(0, detector_noise, canvas.shape).astype(np.float32)
    canvas  = np.clip(canvas, 0.0, 1.0)

    # ── Step 7: Scale bar ─────────────────────────────────────────────────────
    if show_scalebar:
        canvas = _add_scalebar(canvas, magnification, S)

    if return_phase_map:
        return canvas, martensite_phase, actual_fraction
    return canvas, actual_fraction


# ── Scale bar helper (unchanged) ─────────────────────────────────────────────

def _add_scalebar(canvas: np.ndarray, magnification: str, S: int) -> np.ndarray:
    bar_len = int(S * 0.15)
    bar_h   = max(3, int(S * 0.008))
    margin  = int(S * 0.03)
    x0 = S - margin - bar_len
    y0 = S - margin - bar_h - int(S * 0.035)
    x1 = S - margin
    y1 = y0 + bar_h
    canvas[y0:y1, x0:x1] = 1.0
    tick_h = bar_h * 3
    canvas[y0 - tick_h:y1, x0:x0 + max(1, bar_h // 2)] = 1.0
    canvas[y0 - tick_h:y1, x1 - max(1, bar_h // 2):x1] = 1.0
    return canvas


print('generate_microstructure (fractal + fuzzy pipeline) defined.')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Fuzzy Helper 2: Soft grain-boundary blending via Gaussian membership
#
# Replaces hard "paint boundary pixels black" with EDT-based Gaussian
# membership functions, producing the diffuse dark halos seen in real BSE
# images at phase interfaces and grain boundaries.
# ──────────────────────────────────────────────────────────────────────────────

def fuzzy_boundary_blend(
    canvas: np.ndarray,
    ferrite_map: np.ndarray,
    martensite_phase: np.ndarray,
    sigma_gb: float = 3.5,
    sigma_phase: float = 4.0,
    gb_dark: float = 0.05,
    mart_edge_dark: float = 0.03,
) -> np.ndarray:
    """Blend grain-boundary and phase-interface darkening using fuzzy membership.

    For every pixel the membership in the "grain interior" is computed as
        mu = 1 - exp(-d² / (2σ²))
    where d is the EDT distance to the nearest boundary.  Canvas pixels are
    then a weighted mix of their current value and the boundary dark value.

    Parameters
    ----------
    canvas          : float32 (H, W) — current rendered image
    ferrite_map     : int (H, W) — Voronoi grain IDs
    martensite_phase: bool (H, W) — True = martensite pixel
    sigma_gb        : float — transition width at ferrite grain boundaries (px)
    sigma_phase     : float — transition width at martensite/ferrite interface (px)
    gb_dark         : float — dark value at ferrite grain boundaries
    mart_edge_dark  : float — dark value at martensite island edges

    Returns
    -------
    canvas : float32 (H, W), modified in-place and returned
    """
    # ── Ferrite grain boundaries ──────────────────────────────────────────────
    gb_line = grain_boundary_mask(ferrite_map, thickness=1)
    edt_gb  = distance_transform_edt(~gb_line).astype(np.float32)
    mu_ferrite_interior = 1.0 - np.exp(-(edt_gb ** 2) / (2.0 * sigma_gb ** 2 + 1e-8))

    ferrite_px = ~martensite_phase
    canvas[ferrite_px] = (
        mu_ferrite_interior[ferrite_px] * canvas[ferrite_px] +
        (1.0 - mu_ferrite_interior[ferrite_px]) * gb_dark
    )

    # ── Martensite island edges (interior side) ───────────────────────────────
    edt_mart_inner = distance_transform_edt(martensite_phase).astype(np.float32)
    mu_mart_interior = 1.0 - np.exp(-(edt_mart_inner ** 2) / (2.0 * sigma_phase ** 2 + 1e-8))

    mart_px = martensite_phase
    canvas[mart_px] = (
        mu_mart_interior[mart_px] * canvas[mart_px] +
        (1.0 - mu_mart_interior[mart_px]) * mart_edge_dark
    )

    return canvas


print('fuzzy_boundary_blend defined.')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Utility: martensite lath sub-structure within a parent austenite grain
# ──────────────────────────────────────────────────────────────────────────────

def draw_martensite_laths(
    canvas: np.ndarray,
    mask: np.ndarray,
    rng: np.random.Generator,
    n_packets: int = 4,
    laths_per_packet: int = 8,
    lath_width: float = 1.5,
    boundary_value: float = 0.05,
    lath_value_range: tuple = (0.08, 0.22),
) -> np.ndarray:
    """Render lath martensite sub-structure inside `mask` region on `canvas`.
    Returns modified canvas."""
    ys, xs = np.where(mask)
    if len(ys) == 0:
        return canvas

    size = canvas.shape[0]
    cx, cy = xs.mean(), ys.mean()

    # Each packet has a dominant lath direction (3 Bain variants, 24 KS variants → simplified)
    packet_angles = rng.uniform(0, np.pi, n_packets)

    for p_angle in packet_angles:
        # Slightly perturbed lath directions within this packet
        lath_angles = p_angle + rng.normal(0, 0.05, laths_per_packet)
        lath_values = rng.uniform(*lath_value_range, laths_per_packet)

        for angle, val in zip(lath_angles, lath_values):
            # Direction vector of the lath
            dx, dy = np.cos(angle), np.sin(angle)
            # Normal to lath
            nx, ny = -dy, dx

            # Random offset from grain centroid
            extent = max(xs.max() - xs.min(), ys.max() - ys.min()) * 0.5
            offset = rng.uniform(-extent, extent)

            # Signed distance of each pixel in the mask to this lath plane
            dist = (xs - cx) * nx + (ys - cy) * ny - offset
            in_lath = np.abs(dist) < lath_width

            # Draw lath boundary (dark line) and lath interior (slightly lighter)
            lath_pixels = np.zeros(mask.shape, dtype=bool)
            lath_pixels[ys[in_lath], xs[in_lath]] = True
            lath_pixels &= mask

            # Interior of lath – a specific grey tone
            interior = np.zeros(mask.shape, dtype=bool)
            interior_dist = (xs - cx) * nx + (ys - cy) * ny - offset
            in_interior = np.abs(interior_dist) < (lath_width * 0.5)
            interior[ys[in_interior], xs[in_interior]] = True
            interior &= mask

            canvas[lath_pixels] = val
            # Sub-boundary (thin dark line at lath edge)
            boundary_mask = lath_pixels & ~interior
            canvas[boundary_mask] = boundary_value

    return canvas


print('Martensite lath renderer defined.')

## Fractal & fuzzy parameter exploration
Visualise the effect of changing roughness and fuzzy transition width.

In [ ]:
# ── Fractal roughness sweep (ferrite texture) ─────────────────────────────────
roughness_vals = [0.3, 0.55, 0.80]
amplitude_vals = [2.0, 5.0, 10.0]

fig, axes = plt.subplots(len(roughness_vals), len(amplitude_vals),
                          figsize=(3.5 * len(amplitude_vals), 3.5 * len(roughness_vals)))
fig.suptitle('Fractal roughness (rows) × boundary perturbation amplitude (cols)\n'
             'martensite_fraction=0.35', fontsize=12, y=1.01)

for ri, rough in enumerate(roughness_vals):
    for ai, amp in enumerate(amplitude_vals):
        img, _, af = generate_microstructure(
            martensite_fraction=0.35,
            n_ferrite_grains=28,
            n_austenite_grains=15,
            image_size=256,
            seed=ri * 7 + ai * 3 + 11,
            fractal_roughness_ferrite=rough,
            fractal_roughness_boundary=rough,
            boundary_perturbation_amplitude=amp,
            return_phase_map=True,
            show_scalebar=False,
        )
        axes[ri, ai].imshow(img, cmap='gray', vmin=0, vmax=1)
        axes[ri, ai].set_title(f'H={rough}, amp={amp:.0f}px\nM={af:.1%}', fontsize=8)
        axes[ri, ai].axis('off')

axes[0, 0].set_ylabel('', fontsize=8)
for ri, rough in enumerate(roughness_vals):
    axes[ri, 0].set_ylabel(f'H={rough}', fontsize=9, rotation=90, labelpad=4)

plt.tight_layout()
plt.savefig('fractal_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fractal parameter sweep complete.')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Utility: ferrite grain interior texture (slight orientation contrast)
# ──────────────────────────────────────────────────────────────────────────────

def ferrite_grain_texture(
    canvas: np.ndarray,
    mask: np.ndarray,
    base_brightness: float,
    rng: np.random.Generator,
    texture_amplitude: float = 0.04,
    scratch_prob: float = 0.3,
) -> np.ndarray:
    """Fill ferrite grain with orientation-contrast-like texture."""
    ys, xs = np.where(mask)
    if len(ys) == 0:
        return canvas

    # Gentle linear gradient (simulates crystallographic channelling contrast)
    angle = rng.uniform(0, np.pi)
    gradient = np.cos(angle) * xs + np.sin(angle) * ys
    g_norm = (gradient - gradient.min()) / (np.ptp(gradient) + 1e-8)
    brightness = base_brightness + (g_norm - 0.5) * texture_amplitude
    canvas[ys, xs] = brightness

    # Occasional slip/scratch lines
    if rng.random() < scratch_prob:
        scratch_angle = angle + rng.normal(0, 0.1)
        sdx, sdy = np.cos(scratch_angle), np.sin(scratch_angle)
        snx, sny = -sdy, sdx
        cx, cy = xs.mean(), ys.mean()
        offset = rng.uniform(-10, 10)
        dist = np.abs((xs - cx) * snx + (ys - cy) * sny - offset)
        in_scratch = dist < 0.8
        canvas[ys[in_scratch], xs[in_scratch]] = base_brightness - 0.06

    return canvas


print('Ferrite texture renderer defined.')

## Main generator function

In [ ]:
def generate_microstructure(
    martensite_fraction: float = 0.30,
    n_ferrite_grains: int = 40,
    n_austenite_grains: int = 20,
    image_size: int = 512,
    seed: int = 42,
    noise_sigma: float = 3.0,
    detector_noise: float = 0.015,
    boundary_thickness: int = 2,
    magnification: str = '5000x',
    show_scalebar: bool = True,
    return_phase_map: bool = False,
):
    """
    Generate a synthetic SEM BSE-like image of dual-phase martensite/ferrite steel.

    Parameters
    ----------
    martensite_fraction : float
        Target volume fraction of martensite (0–1).
    n_ferrite_grains : int
        Number of ferrite grains in the tessellation.
    n_austenite_grains : int
        Number of prior austenite grains (martensite islands).
    image_size : int
        Output image width/height in pixels.
    seed : int
        RNG seed for reproducibility.
    noise_sigma : float
        Gaussian blur sigma to smooth grain boundaries.
    detector_noise : float
        Amplitude of additive Gaussian noise (SEM detector noise).
    boundary_thickness : int
        Grain boundary line thickness in pixels.
    magnification : str
        Label shown on scale bar.
    show_scalebar : bool
        Whether to overlay a scale bar.
    return_phase_map : bool
        If True, also return a boolean mask where True = martensite.

    Returns
    -------
    img_array : np.ndarray (H, W) float32, range [0, 1]
    phase_map : np.ndarray (H, W) bool  [only if return_phase_map=True]
    actual_fraction : float  measured martensite area fraction
    """
    assert 0.0 <= martensite_fraction <= 1.0, 'martensite_fraction must be in [0, 1]'
    rng = np.random.default_rng(seed)
    S = image_size

    # ── Step 1: Ferrite grain tessellation ───────────────────────────────────
    ferrite_seeds = rng.uniform(0, S, (n_ferrite_grains, 2))
    ferrite_map = voronoi_region_map(ferrite_seeds, S)   # (S, S) int

    # Assign random orientation-dependent brightness per grain
    # Ferrite in BSE: bright phase (~0.55–0.75 normalised)
    ferrite_brightness = rng.uniform(0.55, 0.75, n_ferrite_grains)

    canvas = np.zeros((S, S), dtype=np.float32)

    for gid in range(n_ferrite_grains):
        grain_mask = ferrite_map == gid
        canvas = ferrite_grain_texture(
            canvas, grain_mask, ferrite_brightness[gid], rng
        )

    # ── Step 2: Martensite islands (prior austenite grains) ───────────────────
    # Place austenite seeds – cluster them to mimic real morphology
    # (martensite preferentially forms at ferrite grain boundaries and in
    # carbon-rich regions)
    #
    # Strategy: pick grain boundary pixels preferentially as austenite seed sites
    gb_mask = grain_boundary_mask(ferrite_map, thickness=4)
    gb_ys, gb_xs = np.where(gb_mask)

    if len(gb_ys) < n_austenite_grains:
        # Fall back to random if not enough boundary pixels
        aus_seeds = rng.uniform(0, S, (n_austenite_grains, 2))
    else:
        idx = rng.choice(len(gb_ys), n_austenite_grains, replace=False)
        # Slightly perturb so they spread into grain interiors too
        jitter = rng.normal(0, S * 0.04, (n_austenite_grains, 2))
        aus_seeds = np.column_stack([gb_xs[idx], gb_ys[idx]]).astype(float) + jitter
        aus_seeds = np.clip(aus_seeds, 0, S - 1)

    aus_map = voronoi_region_map(aus_seeds, S)  # (S, S) int

    # ── Step 3: Assign phase to each pixel ────────────────────────────────────
    # Sort austenite grains by their pixel count and assign the largest ones
    # as martensite until we hit the target fraction.
    aus_grain_sizes = np.array([(aus_map == i).sum() for i in range(n_austenite_grains)])
    sorted_ids = np.argsort(-aus_grain_sizes)  # largest first

    total_pixels = S * S
    target_pixels = int(martensite_fraction * total_pixels)

    martensite_phase = np.zeros((S, S), dtype=bool)
    accumulated = 0

    for gid in sorted_ids:
        if accumulated >= target_pixels:
            break
        grain_mask = aus_map == gid
        # Only assign pixels not already claimed; stop when target reached
        remaining_needed = target_pixels - accumulated
        pixel_ys, pixel_xs = np.where(grain_mask)
        if len(pixel_ys) <= remaining_needed:
            martensite_phase[grain_mask] = True
            accumulated += len(pixel_ys)
        else:
            # Partial grain: assign a contiguous sub-region
            # Use a random half-plane cut within the grain
            cx, cy = pixel_xs.mean(), pixel_ys.mean()
            cut_angle = rng.uniform(0, np.pi)
            dx, dy = np.cos(cut_angle), np.sin(cut_angle)
            proj = (pixel_xs - cx) * dx + (pixel_ys - cy) * dy
            order = np.argsort(proj)
            chosen = order[:remaining_needed]
            martensite_phase[pixel_ys[chosen], pixel_xs[chosen]] = True
            accumulated += remaining_needed
            break

    actual_fraction = martensite_phase.sum() / total_pixels

    # ── Step 4: Render martensite lath sub-structure ──────────────────────────
    # Find connected martensite islands and render each one independently
    labeled_mart, n_islands = nd_label(martensite_phase)

    for island_id in range(1, n_islands + 1):
        island_mask = labeled_mart == island_id
        if island_mask.sum() < 20:   # skip tiny specks
            canvas[island_mask] = rng.uniform(0.12, 0.22)
            continue
        # Base martensite BSE intensity (dark phase, ~0.12–0.30)
        base_val = rng.uniform(0.12, 0.28)
        canvas[island_mask] = base_val
        canvas = draw_martensite_laths(canvas, island_mask, rng)

    # ── Step 5: Grain boundaries ──────────────────────────────────────────────
    # Ferrite grain boundaries (dark lines)
    ferrite_gb = grain_boundary_mask(ferrite_map, thickness=boundary_thickness)
    # Austenite/martensite boundary
    mart_boundary = grain_boundary_mask(martensite_phase.astype(int), thickness=boundary_thickness)

    canvas[ferrite_gb & ~martensite_phase] = 0.05  # dark GB on ferrite
    canvas[mart_boundary] = 0.03                   # even darker on martensite edge

    # ── Step 6: Smooth + noise ────────────────────────────────────────────────
    canvas = gaussian_filter(canvas, sigma=noise_sigma * 0.15)
    canvas += rng.normal(0, detector_noise, canvas.shape).astype(np.float32)
    canvas = np.clip(canvas, 0.0, 1.0)

    # ── Step 7: Scale bar overlay ─────────────────────────────────────────────
    if show_scalebar:
        canvas = _add_scalebar(canvas, magnification, S)

    if return_phase_map:
        return canvas, martensite_phase, actual_fraction
    return canvas, actual_fraction


# ── Scale bar helper ──────────────────────────────────────────────────────────

def _add_scalebar(canvas: np.ndarray, magnification: str, S: int) -> np.ndarray:
    """Burn a simple scale bar into the bottom-right corner."""
    bar_len = int(S * 0.15)   # 15% of image width
    bar_h   = max(3, int(S * 0.008))
    margin  = int(S * 0.03)
    x0 = S - margin - bar_len
    y0 = S - margin - bar_h - int(S * 0.035)
    x1 = S - margin
    y1 = y0 + bar_h
    canvas[y0:y1, x0:x1] = 1.0   # white bar
    # Tick marks at ends
    tick_h = bar_h * 3
    canvas[y0 - tick_h:y1, x0:x0 + max(1, bar_h // 2)] = 1.0
    canvas[y0 - tick_h:y1, x1 - max(1, bar_h // 2):x1] = 1.0
    return canvas


print('Generator function defined.')

## Quick test — single image

In [ ]:
img, phase_map, actual_frac = generate_microstructure(
    martensite_fraction=0.30,
    n_ferrite_grains=35,
    n_austenite_grains=18,
    image_size=512,
    seed=7,
    return_phase_map=True,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Synthetic SEM BSE — Martensite {actual_frac:.1%}', fontsize=13)
axes[0].axis('off')

# Phase map overlay
rgb = np.stack([img, img, img], axis=-1)
rgb[phase_map] = [0.85, 0.20, 0.15]   # red tint on martensite
axes[1].imshow(rgb)
red_patch   = mpatches.Patch(color=[0.85, 0.20, 0.15], label=f'Martensite ({actual_frac:.1%})')
grey_patch  = mpatches.Patch(color=[0.65, 0.65, 0.65], label=f'Ferrite ({1-actual_frac:.1%})')
axes[1].legend(handles=[red_patch, grey_patch], loc='lower left', fontsize=10)
axes[1].set_title('Phase map overlay', fontsize=13)
axes[1].axis('off')

plt.tight_layout()
plt.savefig('microstructure_single.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Measured martensite fraction: {actual_frac:.4f}')

## Sweep — varying phase fraction
Generate a grid of images at different martensite volume fractions.

In [ ]:
target_fractions = [0.05, 0.15, 0.30, 0.50, 0.70, 0.90]
n_cols = len(target_fractions)

fig, axes = plt.subplots(2, n_cols, figsize=(3.5 * n_cols, 7.5))
fig.suptitle('Dual-Phase Steel — Martensite/Ferrite Phase Fraction Sweep', fontsize=14, y=1.01)

for col, frac in enumerate(target_fractions):
    img, pm, af = generate_microstructure(
        martensite_fraction=frac,
        n_ferrite_grains=30,
        n_austenite_grains=20,
        image_size=256,
        seed=col * 13 + 5,
        return_phase_map=True,
        show_scalebar=False,
    )

    # Row 0: greyscale SEM-like
    axes[0, col].imshow(img, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'M={frac:.0%}\n(actual {af:.1%})', fontsize=9)
    axes[0, col].axis('off')

    # Row 1: phase map
    rgb = np.stack([img, img, img], axis=-1)
    rgb[pm] = [0.85, 0.20, 0.15]
    axes[1, col].imshow(rgb)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('BSE image', fontsize=9)
axes[1, 0].set_ylabel('Phase map', fontsize=9)

plt.tight_layout()
plt.savefig('microstructure_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sweep complete.')

## Batch generation — produce a labelled dataset
Generate N images per target fraction and save them as PNG + a CSV manifest.

In [ ]:
import os, csv
from pathlib import Path


def batch_generate(
    output_dir: str = 'generated_microstructures',
    fractions: list = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70],
    n_per_fraction: int = 5,
    image_size: int = 512,
    base_seed: int = 0,
):
    """
    Generate `n_per_fraction` images at each target martensite fraction.
    Saves PNGs and a manifest CSV to `output_dir`.
    """
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    manifest_path = out / 'manifest.csv'
    rows = []
    total = len(fractions) * n_per_fraction
    done = 0

    for frac in fractions:
        for i in range(n_per_fraction):
            seed = base_seed + done
            img, phase_map, af = generate_microstructure(
                martensite_fraction=frac,
                n_ferrite_grains=rng_n(seed, 25, 50),
                n_austenite_grains=rng_n(seed + 1, 12, 30),
                image_size=image_size,
                seed=seed,
                noise_sigma=np.random.default_rng(seed).uniform(1.5, 4.0),
                return_phase_map=True,
                show_scalebar=True,
            )

            fname = f'micro_{frac:.2f}_{i:03d}.png'
            # Convert float32 → uint8 greyscale
            pil_img = Image.fromarray((img * 255).astype(np.uint8), mode='L')
            pil_img.save(out / fname)

            rows.append({
                'filename': fname,
                'target_martensite_fraction': frac,
                'actual_martensite_fraction': round(af, 5),
                'seed': seed,
                'image_size': image_size,
            })
            done += 1
            if done % 5 == 0 or done == total:
                print(f'  {done}/{total} images written...', end='\r')

    with open(manifest_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    print(f'\nDone. {total} images + manifest saved to: {out.resolve()}')
    return manifest_path


def rng_n(seed, lo, hi):
    return int(np.random.default_rng(seed).integers(lo, hi))


print('Batch generator defined.\n'
      'Call batch_generate() below to produce the dataset.')

In [ ]:
# ── Run batch generation (adjust parameters as needed) ────────────────────────
manifest = batch_generate(
    output_dir='generated_microstructures',
    fractions=[0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90],
    n_per_fraction=5,
    image_size=512,
    base_seed=1000,
)

## Inspect generated dataset

In [ ]:
import pandas as pd

df = pd.read_csv(manifest)
print(df.to_string(index=False))

# Accuracy of fraction targeting
df['error'] = (df['actual_martensite_fraction'] - df['target_martensite_fraction']).abs()
print(f'\nMean |target − actual| fraction: {df["error"].mean():.4f}')
print(f'Max  |target − actual| fraction: {df["error"].max():.4f}')

# Quick histogram
fig, ax = plt.subplots(figsize=(7, 3))
ax.scatter(df['target_martensite_fraction'], df['actual_martensite_fraction'],
           alpha=0.7, edgecolors='k', linewidths=0.5)
ax.plot([0, 1], [0, 1], 'r--', lw=1, label='Perfect targeting')
ax.set_xlabel('Target martensite fraction')
ax.set_ylabel('Actual martensite fraction')
ax.set_title('Phase fraction targeting accuracy')
ax.legend()
plt.tight_layout()
plt.show()

## Interactive single-image configurator
Run this cell to explore microstructures interactively via parameter sliders (requires `ipywidgets`).

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    def _interactive_plot(
        martensite_fraction, n_ferrite_grains, n_austenite_grains, seed, noise_sigma
    ):
        img, pm, af = generate_microstructure(
            martensite_fraction=martensite_fraction,
            n_ferrite_grains=n_ferrite_grains,
            n_austenite_grains=n_austenite_grains,
            image_size=384,
            seed=seed,
            noise_sigma=noise_sigma,
            return_phase_map=True,
            show_scalebar=True,
        )
        fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
        axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
        axes[0].set_title(f'BSE image — M={af:.1%}', fontsize=12)
        axes[0].axis('off')
        rgb = np.stack([img, img, img], axis=-1)
        rgb[pm] = [0.85, 0.20, 0.15]
        axes[1].imshow(rgb)
        axes[1].set_title('Phase map  (red=martensite)', fontsize=12)
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()

    widgets.interact(
        _interactive_plot,
        martensite_fraction=widgets.FloatSlider(value=0.30, min=0.02, max=0.95, step=0.01,
                                                description='M fraction'),
        n_ferrite_grains=widgets.IntSlider(value=35, min=10, max=80, step=5,
                                           description='Ferrite grains'),
        n_austenite_grains=widgets.IntSlider(value=18, min=5, max=50, step=1,
                                             description='Aus. grains'),
        seed=widgets.IntSlider(value=42, min=0, max=500, step=1, description='Seed'),
        noise_sigma=widgets.FloatSlider(value=2.0, min=0.5, max=8.0, step=0.5,
                                        description='Blur σ'),
    )
except ImportError:
    print('ipywidgets not installed — run: pip install ipywidgets')
    print('Using static call instead:\n')
    img, pm, af = generate_microstructure(
        martensite_fraction=0.45,
        n_ferrite_grains=40,
        n_austenite_grains=22,
        image_size=384,
        seed=99,
        return_phase_map=True,
    )
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
    axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title(f'BSE image — M={af:.1%}', fontsize=12)
    axes[0].axis('off')
    rgb = np.stack([img, img, img], axis=-1)
    rgb[pm] = [0.85, 0.20, 0.15]
    axes[1].imshow(rgb)
    axes[1].set_title('Phase map  (red=martensite)', fontsize=12)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Martensite fraction: {af:.4f}')